In [1]:
import os
import sys
import time
sys.path.append("..")

from tqdm import tqdm
import pandas as pd
import matplotlib.pyplot as plt

from CalixVolApp.calculation.calculation import MoleculeFileReader, JsonVDWRadiusProvider, ConvexHullVolumeEstimator, MoleculeVolumeCalculator
from CalixVolApp.utils.paths import get_project_path

# Grid Search

In [ ]:
resolutions = [0.02, 0.1, 1.0]
molecules = range(1, 15)

df_vol = pd.DataFrame(index=molecules)
df_time = pd.DataFrame(index=molecules)

vdw_file = os.path.join(get_project_path(), 'CalixVolApp', 'data', 'vdw', 'vdw_radius.json')
reader = MoleculeFileReader()
vdw_provider = JsonVDWRadiusProvider(vdw_file)
estimator = ConvexHullVolumeEstimator()
calculator = MoleculeVolumeCalculator(reader, vdw_provider, estimator)

for res in tqdm(resolutions, desc="Resolutions"):
    vols, times = [], []
    for mol in tqdm(molecules, desc="Molecule", leave=False):
        molecule_file = os.path.join(
            get_project_path(), 'CalixVolApp', 'data', 'molecules', 'txt_calix', f'{mol}.txt'
        )

        t0 = time.perf_counter()
        volume_total, volume_atom, volume_cavity = calculator.calculate(
            molecule_file, grid_resolution=res
        )
        elapsed = time.perf_counter() - t0

        vols.append(volume_cavity / 2 if mol == 14 else volume_cavity)
        times.append(elapsed)

    df_vol[res] = vols
    df_time[res] = times

df_vol.to_csv(os.path.join(get_project_path(), 'results', 'CaviDAC', 'vol_vs_grid_res.csv'), index=False)
df_time.to_csv(os.path.join(get_project_path(), 'results', 'CaviDAC', 'time_s_vs_grid_res.csv'), index=False)

# Calculate Volume for all molecules

In [2]:
molecules, grid_res, volumes = [], [], []

for mol in range(1, 15):
    vdw_file = os.path.join(get_project_path(), 'CalixVolApp', 'data', 'vdw', 'vdw_radius.json')
    molecule_file = os.path.join(get_project_path(), 'CalixVolApp', 'data', 'molecules', 'txt_calix', f'{mol}.txt')

    reader = MoleculeFileReader()
    vdw_provider = JsonVDWRadiusProvider(vdw_file)
    estimator = ConvexHullVolumeEstimator()
    calculator = MoleculeVolumeCalculator(reader, vdw_provider, estimator)

    volume_total, volume_atom, volume_cavity = calculator.calculate(molecule_file, grid_resolution=0.1)
    
    if mol == 14:
        print(f"- - - - - - - - - - - {mol} - - - - - - - - - - -")
        print(volume_cavity / 2)
        volumes.append(volume_cavity / 2)
    else:
        print(f"- - - - - - - - - - - {mol} - - - - - - - - - - -")
        print(volume_cavity)
        volumes.append(volume_cavity)

    molecules.append(mol+1)
    grid_res.append(0.1)

df_res = pd.DataFrame({'Molecule': molecules, 'Grid_Resolution': grid_res, 'Cavity_Volume': volumes})
df_res.to_csv(os.path.join(get_project_path(), 'results', 'CaviDAC', 'calc_volumes_cavidac.csv'), index=False)

- - - - - - - - - - - 1 - - - - - - - - - - -
116.44426568242167
- - - - - - - - - - - 2 - - - - - - - - - - -
81.46876176438906
- - - - - - - - - - - 3 - - - - - - - - - - -
138.29822271480273
- - - - - - - - - - - 4 - - - - - - - - - - -
61.62286670380206
- - - - - - - - - - - 5 - - - - - - - - - - -
117.74454796667607
- - - - - - - - - - - 6 - - - - - - - - - - -
120.91872669588457
- - - - - - - - - - - 7 - - - - - - - - - - -
121.17318531842136
- - - - - - - - - - - 8 - - - - - - - - - - -
97.57717014438052
- - - - - - - - - - - 9 - - - - - - - - - - -
88.25843034704891
- - - - - - - - - - - 10 - - - - - - - - - - -
63.64243881183776
- - - - - - - - - - - 11 - - - - - - - - - - -
117.96011817702362
- - - - - - - - - - - 12 - - - - - - - - - - -
59.603455625203594
- - - - - - - - - - - 13 - - - - - - - - - - -
127.04238971398124
- - - - - - - - - - - 14 - - - - - - - - - - -
81.54521081635126


In [5]:
df_res

,Molecule,Grid_Resolution,Cavity_Volume
0,1,0.1,116.444266
1,2,0.1,81.468762
2,3,0.1,138.298223
3,4,0.1,61.622867
4,5,0.1,117.744548
5,6,0.1,120.918727
6,7,0.1,121.173185
7,8,0.1,97.577170
8,9,0.1,88.258430
9,10,0.1,63.642439
